<a href="https://www.kaggle.com/code/ianpetrustan/worldcup-data-engineering-exploratory-analysis?scriptVersionId=327270440" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Cleaning and exploring, feature engineering

* understanding the data
* cleaning and feature engineering
* data analysis, data auditing
* **preparing the data for downstream modelling/analysis**

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/isfakiqbalchowdhuruy/fifa-mens-world-cup-dataset-1970-2022/fifa_wc_mens_match_dataset_1970_2022.csv


In [2]:
## Set some global stuff
pd.set_option('display.max_columns', None)


In [3]:
filepath = '/kaggle/input/datasets/isfakiqbalchowdhuruy/fifa-mens-world-cup-dataset-1970-2022/fifa_wc_mens_match_dataset_1970_2022.csv'

## read file 
wc_full = pd.read_csv(filepath)

## basic stats ------------------------------------

def basic_stats(wc_full):
    print(f'file size: {wc_full.shape}')
    
    ## columns 
    print(f'cols: {wc_full.columns}')
    
    ## duration 
    print(f'date range: {wc_full.match_date.min()} to {wc_full.match_date.max()}')

basic_stats(wc_full)

file size: (1322, 70)
cols: Index(['tournament_name', 'stage_name', 'group_name', 'group_stage',
       'knockout_stage', 'replayed', 'replay', 'match_date', 'match_time',
       'stadium_name', 'city_name', 'country_name', 'team_name', 'team_code',
       'opponent_name', 'opponent_code', 'home_team', 'away_team', 'goals_for',
       'goals_against', 'extra_time', 'penalty_shootout', 'penalties_for',
       'penalties_against', 'result', 'is_host', 'yellow_cards', 'red_cards',
       'possession', 'shots', 'shots_on_target', 'passes_completed',
       'passes_attempted', 'corners', 'fouls', 'team_prior_matches',
       'team_prior_win_rate', 'team_prior_goals_scored_avg',
       'team_prior_goals_conceded_avg', 'opp_prior_matches',
       'opp_prior_win_rate', 'opp_prior_goals_scored_avg',
       'opp_prior_goals_conceded_avg', 'h2h_prior_matches',
       'h2h_prior_win_rate', 'team_curr_form_pts_avg',
       'team_curr_goals_scored_avg', 'team_curr_goals_conceded_avg',
       'opp_cu

In [4]:
## How many seasons of wc?
def further_stats(wc_full):
    print(f'Number of seasons: {wc_full.tournament_name.nunique()}')

    ## How many groups for group stages?
    print('Group stages per tournament: ')
    print(wc_full.groupby('tournament_name')['group_name'].nunique())

    ## All stages
    print(wc_full.groupby('tournament_name')['stage_name'].unique())

further_stats(wc_full)

Number of seasons: 14
Group stages per tournament: 
tournament_name
1970 FIFA Men's World Cup    4
1974 FIFA Men's World Cup    7
1978 FIFA Men's World Cup    7
1982 FIFA Men's World Cup    7
1986 FIFA Men's World Cup    7
1990 FIFA Men's World Cup    7
1994 FIFA Men's World Cup    7
1998 FIFA Men's World Cup    9
2002 FIFA Men's World Cup    9
2006 FIFA Men's World Cup    9
2010 FIFA Men's World Cup    9
2014 FIFA Men's World Cup    9
2018 FIFA Men's World Cup    9
2022 FIFA Men's World Cup    9
Name: group_name, dtype: int64
tournament_name
1970 FIFA Men's World Cup    [group stage, quarter-finals, semi-finals, thi...
1974 FIFA Men's World Cup    [group stage, second group stage, third-place ...
1978 FIFA Men's World Cup    [group stage, second group stage, third-place ...
1982 FIFA Men's World Cup    [group stage, second group stage, semi-finals,...
1986 FIFA Men's World Cup    [group stage, round of 16, quarter-finals, sem...
1990 FIFA Men's World Cup    [group stage, round of 16, 

## Understanding the dataset

In [5]:
## peek 
pd.set_option('display.max_columns', None)
wc_full.head()

## understanding: each row focuses on a team and its opposition. the outcome relates to this team 

,tournament_name,stage_name,group_name,group_stage,knockout_stage,replayed,replay,match_date,match_time,stadium_name,city_name,country_name,team_name,team_code,opponent_name,opponent_code,home_team,away_team,goals_for,goals_against,extra_time,penalty_shootout,penalties_for,penalties_against,result,is_host,yellow_cards,red_cards,possession,shots,shots_on_target,passes_completed,passes_attempted,corners,fouls,team_prior_matches,team_prior_win_rate,team_prior_goals_scored_avg,team_prior_goals_conceded_avg,opp_prior_matches,opp_prior_win_rate,opp_prior_goals_scored_avg,opp_prior_goals_conceded_avg,h2h_prior_matches,h2h_prior_win_rate,team_curr_form_pts_avg,team_curr_goals_scored_avg,team_curr_goals_conceded_avg,opp_curr_form_pts_avg,opp_curr_goals_scored_avg,opp_curr_goals_conceded_avg,possession_h1,possession_h2,shots_h1,shots_h2,shots_on_target_h1,shots_on_target_h2,passes_completed_h1,passes_completed_h2,passes_attempted_h1,passes_attempted_h2,corners_h1,corners_h2,fouls_h1,fouls_h2,yellow_cards_h1,yellow_cards_h2,red_cards_h1,red_cards_h2,outcome
0,1970 FIFA Men's World Cup,group stage,Group 1,1,0,0,0,1970-05-31,12:00,Estadio Azteca,Mexico City,Mexico,Mexico,MEX,Soviet Union,SUN,1,0,0,0,0,0,0,0,draw,1,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17,0.058824,0.764706,2.705882,15,0.533333,1.600000,1.266667,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,0.0,0.0,Draw
1,1970 FIFA Men's World Cup,group stage,Group 1,1,0,0,0,1970-05-31,12:00,Estadio Azteca,Mexico City,Mexico,Soviet Union,SUN,Mexico,MEX,0,1,0,0,0,0,0,0,draw,0,4,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15,0.533333,1.600000,1.266667,17,0.058824,0.764706,2.705882,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,1.0,0.0,0.0,Draw
2,1970 FIFA Men's World Cup,group stage,Group 2,1,0,0,0,1970-06-03,16:00,La Bombonera,Toluca,Mexico,Sweden,SWE,Italy,ITA,0,1,0,1,0,0,0,0,lose,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16,0.500000,2.375000,2.187500,20,0.600000,1.900000,1.100000,1,1.0,0.0,0.0,0.0,3.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,0.0,0.0,Loss
3,1970 FIFA Men's World Cup,group stage,Group 3,1,0,0,0,1970-06-03,16:00,Estadio Jalisco,Guadalajara,Mexico,Brazil,BRA,Czechoslovakia,CSK,1,0,4,1,0,0,0,0,win,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32,0.625000,2.625000,1.312500,19,0.421053,1.578947,1.526316,4,0.5,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,0.0,0.0,Win
4,1970 FIFA Men's World Cup,group stage,Group 3,1,0,0,0,1970-06-06,16:00,Estadio Jalisco,Guadalajara,Mexico,Romania,ROU,Czechoslovakia,CSK,1,0,2,1,0,0,0,0,win,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,0.200000,1.600000,2.400000,19,0.421053,1.578947,1.526316,1,0.0,0.0,0.0,1.0,0.0,1.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,0.0,0.0,Win


In [6]:
def extract_wc_year(wc_full):
    wc_full['wc_year'] = wc_full.tournament_name.str.split().str[0].astype(int)
    return wc_full

wc_full = extract_wc_year(wc_full)

### What does each row signify?

In [7]:
## look at one tournament and one group stage 
first_season = wc_full.tournament_name == "1970 FIFA Men's World Cup"
first_group = wc_full.group_name == 'Group 1'

wc_full[first_season&first_group].sort_values(['match_date', 'match_time'])[['match_date', 'match_time', 'team_name', 'opponent_name', 'outcome']]

,match_date,match_time,team_name,opponent_name,outcome
0,1970-05-31,12:00,Mexico,Soviet Union,Draw
1,1970-05-31,12:00,Soviet Union,Mexico,Draw
6,1970-06-07,12:00,El Salvador,Mexico,Loss
10,1970-06-10,16:00,Soviet Union,El Salvador,Win
12,1970-06-11,16:00,Mexico,Belgium,Win
13,1970-06-11,16:00,Belgium,Mexico,Loss


We see that each game is defined by the tournament, match_date, match_time, and teams

For the Mexico-SU game, there are two rows, one for each team's POV. 
However, for the El Salvador-Mexico game, there is only one row, from ES's POV. 

Hence, we need a deduplication step to standardise across the whole dataset

In [8]:
## We can check across the whole dataset how many games have 2 rows (one for each team)
## and how many only have one:

def create_game_id(wc_full):
    '''
    Create human-readable ID for each game
    '''
    ## symmetrize/ sort between team and opponent 
    ## axis = 1 to sort horizontally 
    teams_symmetric = np.sort(wc_full[['team_name', 'opponent_name']], axis = 1)

    ## composite id for each game 
    game_id = wc_full.tournament_name + '|'+ \
                wc_full.stage_name + '|'+ \
                wc_full.match_date + '|'+ \
                wc_full.match_time + '|'+ \
                teams_symmetric[:,0] + '|'+ \
                teams_symmetric[:,1]
    ## create col 
    wc_full['game_id'] = game_id
    
    return wc_full

def create_surrogate_match_id(df):
    '''
    Creates numerical ID for each game 
    '''
    df['match_id'] = pd.factorize(df.game_id)[0] + 1
    return df



def check_repeated_single_game_distr(wc_full):
    game_id = create_game_id(wc_full)
    ## Get distribution of repeated games vs single games 
    return game_id.value_counts().value_counts().reset_index(name = 'n_games').\
                    rename(columns = {'count': 'row_per_game'})
    

check_repeated_single_game_distr(wc_full)

## create game and match id 
wc_full = create_game_id(wc_full)
wc_full = create_surrogate_match_id(wc_full)


## Result
# count
# 2:    612 ##-- 612 unique games have both rows present 
# 1:    98 ##-- 98 unique games only have 1 row present 
# Name: count, dtype: int64

## the single games are in the minority -- we can investigate further (data audit )
    

In [9]:
## Create column for single row games

## single row games 
single_match_ids = wc_full.match_id.value_counts().index[(wc_full.match_id.value_counts() == 1)]

wc_full['single_row'] = 0
wc_full.loc[wc_full.match_id.isin(single_match_ids), 'single_row'] = 1

## proportion of unique games that are single row 
wc_full.groupby('wc_year')['single_row'].sum()/wc_full.groupby('wc_year')['match_id'].nunique()

wc_year
1970    0.421053
1974    0.454545
1978    0.400000
1982    0.465116
1986    0.361702
1990    0.270833
1994    0.115385
1998    0.171875
2002    0.000000
2006    0.000000
2010    0.000000
2014    0.000000
2018    0.000000
2022    0.000000
dtype: float64

We see that the single match rows only appear before year 2000 --> data becomes 'cleaner' over time, even before 2000, number of single row games was dropping

## Populating missing 'other side' games, for the single row games

In [10]:
wc_full.columns

Index(['tournament_name', 'stage_name', 'group_name', 'group_stage',
       'knockout_stage', 'replayed', 'replay', 'match_date', 'match_time',
       'stadium_name', 'city_name', 'country_name', 'team_name', 'team_code',
       'opponent_name', 'opponent_code', 'home_team', 'away_team', 'goals_for',
       'goals_against', 'extra_time', 'penalty_shootout', 'penalties_for',
       'penalties_against', 'result', 'is_host', 'yellow_cards', 'red_cards',
       'possession', 'shots', 'shots_on_target', 'passes_completed',
       'passes_attempted', 'corners', 'fouls', 'team_prior_matches',
       'team_prior_win_rate', 'team_prior_goals_scored_avg',
       'team_prior_goals_conceded_avg', 'opp_prior_matches',
       'opp_prior_win_rate', 'opp_prior_goals_scored_avg',
       'opp_prior_goals_conceded_avg', 'h2h_prior_matches',
       'h2h_prior_win_rate', 'team_curr_form_pts_avg',
       'team_curr_goals_scored_avg', 'team_curr_goals_conceded_avg',
       'opp_curr_form_pts_avg', 'opp_curr_

In [11]:
## verify definition of 'possession' -- should sum to 100% across both sides of the same game 
wc_full[wc_full.single_row!=1].groupby('game_id')['possession'].sum().value_counts()

possession
100.0    383
0.0      228
99.0       1
Name: count, dtype: int64

In [12]:
## see some examples 
wc_full[wc_full.single_row != 1].sort_values('match_id').head()

## check if these columns are the same across both sides: yellow_cards, red_cards, possession, shots, shots_on
## yellow cards, red cards, ... columns are all different for each side! hence every row captures unique information (apart from match information, e.g. match date, match time, etc)

## means that for missing 'other side' games, most of the information is missing, apart from game outcome, goals for and against, is host, 
swap_pairs = {
    # identities
    'team_name': 'opponent_name',
    'team_code': 'opponent_code',
    # 'home_team': 'away_team', ## we only keep home team column and will take 1 - home_team for the opposite side

    # scoreline
    'goals_for': 'goals_against',
    'penalties_for': 'penalties_against',

    # historical features
    'team_prior_matches': 'opp_prior_matches',
    'team_prior_win_rate': 'opp_prior_win_rate',
    'team_prior_goals_scored_avg': 'opp_prior_goals_scored_avg',
    'team_prior_goals_conceded_avg': 'opp_prior_goals_conceded_avg',

    # tournament form
    'team_curr_form_pts_avg': 'opp_curr_form_pts_avg',
    'team_curr_goals_scored_avg': 'opp_curr_goals_scored_avg',
    'team_curr_goals_conceded_avg': 'opp_curr_goals_conceded_avg'
}

outcome_map = {
    'Win': 'Loss',
    'Loss': 'Win',
    'Draw': 'Draw'
}

swap_pair_cols = [x for x,y in swap_pairs.items()] + [y for x,y in swap_pairs.items()]
# swap_pair_cols

stat_cols = [
    'yellow_cards',
    'red_cards',
    'possession',
    'shots',
    'shots_on_target',
    'passes_completed',
    'passes_attempted',
    'corners',
    'fouls',

    'possession_h1',
    'possession_h2',
    'shots_h1',
    'shots_h2',
    'shots_on_target_h1',
    'shots_on_target_h2',
    'passes_completed_h1',
    'passes_completed_h2',
    'passes_attempted_h1',
    'passes_attempted_h2',
    'corners_h1',
    'corners_h2',
    'fouls_h1',
    'fouls_h2',
    'yellow_cards_h1',
    'yellow_cards_h2',
    'red_cards_h1',
    'red_cards_h2'
]



In [13]:
## for single game matches, check if these information are missing or now 
wc_full[wc_full.single_row == 1][swap_pair_cols].isna().sum()

team_name                        0
team_code                        0
goals_for                        0
penalties_for                    0
team_prior_matches               0
team_prior_win_rate              0
team_prior_goals_scored_avg      0
team_prior_goals_conceded_avg    0
team_curr_form_pts_avg           0
team_curr_goals_scored_avg       0
team_curr_goals_conceded_avg     0
opponent_name                    0
opponent_code                    0
goals_against                    0
penalties_against                0
opp_prior_matches                0
opp_prior_win_rate               0
opp_prior_goals_scored_avg       0
opp_prior_goals_conceded_avg     0
opp_curr_form_pts_avg            0
opp_curr_goals_scored_avg        0
opp_curr_goals_conceded_avg      0
dtype: int64

In [14]:
## create the opposite side rows by flipping cols 

def create_other_side(wc_full):
    ## create copy of one side games
    single_rows = wc_full.loc[wc_full['single_row']].copy()

    ## check for single row games
    if len(single_rows) == 0:
        print("No single row games")
        return wc_full

    ## continue     
    mirrors = single_rows.copy()
    
    for left, right in swap_pairs.items():
    
        ## hold the original info temp 
        temp = mirrors[left].copy()
    
        ## swap to the opposite info (we already saved the original)
        mirrors[left] = mirrors[right]
        ## add original info to the new side 
        mirrors[right] = temp

    ## reverse outcomes 
    mirrors['outcome'] = mirrors['outcome'].map(outcome_map)

    ## host column 
    mirrors['is_host'] = (
    mirrors['team_name']
    == mirrors['country_name'])

    ## create nan for stat_cols since cannot be inferred
    mirrors[stat_cols] = np.nan

    ## label generated rows 
    mirrors['generated_row'] = True
    wc_full['generated_row'] = False

    wc_team = pd.concat(
    [wc_full, mirrors],
    ignore_index=True
)

In [15]:
(wc_full.country_name == wc_full.team_name)


0        True
1       False
2       False
3       False
4       False
        ...  
1317    False
1318    False
1319    False
1320    False
1321    False
Length: 1322, dtype: bool

In [16]:
## Number of games for each tourny 
games_per_tourny = wc_full.groupby('tournament_name').size().reset_index().rename(columns = {0: 'n_games'})

In [17]:
# ## Number of teams for each tourny 
# home_teams = wc_full[['tournament_name', 'team_name']]
# away_teams = wc_full[['tournament_name', 'opponent_name']].rename(
#     columns = {'opponent_name': 'team_name'})

# teams_per_tourny = pd.concat([home_teams, away_teams], 
#          axis = 0).reset_index(drop = 1).groupby('tournament_name')['team_name'].nunique().reset_index()
# teams_per_tourny = teams_per_tourny.rename(columns = {'team_name': 'n_teams'})

# ## combine and plot 
# n_games_teams_agg = games_per_tourny.merge(teams_per_tourny)

# n_games_teams_agg['year'] = n_games_teams_agg.tournament_name.str.split().str[0]
# from matplotlib.ticker import MaxNLocator

# fig, ax1 = plt.subplots(figsize=(12, 5))

# # Matches
# ax1.plot(
#     n_games_teams_agg['year'],
#     n_games_teams_agg['n_games'],
#     marker='o',
#     color='tab:blue',
#     label='Matches'
# )
# ax1.set_ylabel('Number of Matches', color='tab:blue')
# ax1.tick_params(axis='y', labelcolor='tab:blue')

# # Teams
# ax2 = ax1.twinx()

# ax2.plot(
#     n_games_teams_agg['year'],
#     n_games_teams_agg['n_teams'],
#     marker='s',
#     color='tab:red',
#     label='Teams'
# )
# ax2.set_ylabel('Number of Teams', color='tab:red')
# ax2.set_ylim(0, n_games_teams_agg.n_teams.max() + 5)
# ax2.tick_params(axis='y', labelcolor='tab:red')

# # Force integer ticks on team axis
# ax2.yaxis.set_major_locator(MaxNLocator(integer=True))

# ax1.set_xlabel('Year')
# ax1.set_title('World Cup Expansion Over Time')

# plt.tight_layout()
# plt.show()

In [18]:
## cleaning data types 
pd.set_option('display.max_rows', None)
print(wc_full.dtypes.reset_index().rename(columns = {'index': 'col', 0:'dtype'}).head())
pd.set_option('display.max_rows', 30)


               col   dtype
0  tournament_name  object
1       stage_name  object
2       group_name  object
3      group_stage   int64
4   knockout_stage   int64


## Cleaning steps

In [19]:
## convert date and time to proper formats

## just date 
wc_full.match_date = pd.to_datetime(wc_full.match_date)

## richer datetime col 
wc_full['match_datetime'] = pd.to_datetime(
    wc_full['match_date'].astype(str) + ' ' + wc_full['match_time']
)

## check
wc_full.match_datetime.head()

## match_date can be dropped 

0   1970-05-31 12:00:00
1   1970-05-31 12:00:00
2   1970-06-03 16:00:00
3   1970-06-03 16:00:00
4   1970-06-06 16:00:00
Name: match_datetime, dtype: datetime64[ns]

In [20]:
## drop redundant columns 

## we notice that there is both a 'home' and 'away' col. lets ensure they are redundant (dummy variable trap) and drop one 
## i.e. if they all do indeed sum to 1, then we can safely drop the second column
if all(wc_full[['home_team', 'away_team']].sum(axis = 1) == 1):
    print('dropping away team')
    wc_full = wc_full.drop(columns = 'away_team')
    print(wc_full.shape[1])

dropping away team
74
